# **Method (02) tf.data API**
- Modern
- Most powerful + flexible pipeline system
- Supports:
  - large datasets
  - streaming
  - performance optimization (prefetch, cache, map)

In [1]:
import tensorflow as tf
import numpy as np

In [26]:
# laod data as file paths(strings) into a tf dataset
images = tf.data.Dataset.list_files("data/*/*", shuffle=True, seed=42)  # shuffling is important

In [3]:
len(images)

2152

In [4]:
for i in images.take(3):
    test_path = i
    print(i)

tf.Tensor(b'data\\Potato___Late_blight\\0f6eac3b-d674-4c4d-ab3a-88689feec07f___RS_LB 5232.JPG', shape=(), dtype=string)
tf.Tensor(b'data\\Potato___Late_blight\\3221ccec-f264-4046-8252-77a8a357562b___RS_LB 4637.JPG', shape=(), dtype=string)
tf.Tensor(b'data\\Potato___Late_blight\\7260ba74-3187-4a62-8962-cbea92def2f2___RS_LB 4769.JPG', shape=(), dtype=string)


In [7]:
import os

class_names = os.listdir("data")
class_names = tf.constant(class_names)
class_names

<tf.Tensor: shape=(3,), dtype=string, numpy=
array([b'Potato___Early_blight', b'Potato___healthy',
       b'Potato___Late_blight'], dtype=object)>

In [8]:
def get_label(path):
    label_str = tf.strings.split(path, os.path.sep)[-2]
    label = tf.where(class_names==label_str)[0][0]
    return tf.cast(label, tf.int32)

In [9]:
test_label = get_label(test_path)
test_label

<tf.Tensor: shape=(), dtype=int32, numpy=2>

## Data splitting

In [10]:
image_count = len(images)
print(image_count)

train_size = int(image_count*0.7)
print(train_size)

val_size = int(image_count*0.2)
print(val_size)

test_size = int(image_count*0.1)
print(test_size)

2152
1506
430
215


In [12]:
train_images = images.take(train_size)
val_images = images.skip(train_size).take(val_size)
test_images = images.skip(train_size).skip(val_size)

print(len(train_images), len(val_images), len(test_images))

1506 430 216


# Image processing
- resizing and rescaling must have in CNN model other wise always we have to do resizing and rescaling before feed image to model in production applications

In [32]:
def process_train_image(path):
    label = get_label(path)

    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3)
    img.set_shape([None, None, 3])
    #  move to cnn model as a layer
    img = tf.image.resize(img, [128, 128]) # does not change the channel dimension at all — it only changes height and width
    #  move to cnn model as a layer
    img = img/255 # scaling 0-1

    # augmentation
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)

    return img, label

In [15]:
def process_val_test_image(path):
    label = get_label(path)

    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [128, 128]) # does not change the channel dimension at all — it only changes height and width
    img = img/255 # scaling 0-1
    
    return img, label

In [31]:
# all ds are now TF Dataset and each item is a tensors

train_ds = train_images.map(process_train_image) 
test_ds = test_images.map(process_val_test_image)
val_ds = val_images.map(process_val_test_image)

In [17]:
len(train_ds)

1506

## Batching

In [19]:
train_ds = train_ds.batch(32)
test_ds = test_ds.batch(32)
val_ds = val_ds.batch(32)

In [20]:
type(train_ds)

tensorflow.python.data.ops.batch_op._BatchDataset

In [28]:
len(train_ds)

48

In [29]:
# first batch of Dataset
# iterate thorugh tuples each tuple has two batches as (img_batch, label_batch) in size 32

for i in train_ds.take(3):
    print(i[0].shape, i[1].numpy()) # since large size of mtrices check shape 

(32, 128, 128, 3) [2 1 2 0 2 0 2 2 2 2 0 0 2 0 0 2 0 2 2 2 2 2 2 0 1 0 0 1 0 0 0 0]
(32, 128, 128, 3) [2 1 2 0 2 0 2 0 0 0 1 0 0 0 2 2 0 1 0 2 2 0 0 2 2 0 2 1 0 2 0 0]
(32, 128, 128, 3) [0 2 2 2 0 0 0 2 0 2 0 0 2 1 0 2 0 2 2 2 2 2 2 2 2 0 2 1 2 0 0 0]


In [30]:
for i in val_ds.take(3):
    print(i[0].shape, i[1].numpy()) # since large size of mtrices check shape 

(32, 128, 128, 3) [2 1 0 1 2 2 2 0 0 0 0 0 0 0 2 1 2 0 2 2 2 2 2 0 2 2 0 0 2 0 2 2]
(32, 128, 128, 3) [2 0 2 2 2 1 0 2 0 0 0 2 0 2 2 1 2 2 0 1 0 2 0 2 0 2 0 2 2 0 2 2]
(32, 128, 128, 3) [0 1 1 2 0 1 2 2 0 0 2 2 2 0 0 2 0 2 2 2 0 2 2 2 2 2 2 0 1 2 2 0]


In [25]:
for i in test_ds.take(3):
    print(i[0].shape, i[1].numpy()) # since large size of mtrices check shape 

(32, 128, 128, 3) [2 2 2 0 2 0 2 2 2 2 2 0 0 0 2 2 2 2 2 2 2 2 2 0 2 2 0 2 0 2 1 0]
(32, 128, 128, 3) [0 0 0 2 2 0 0 2 2 0 0 2 0 0 0 2 0 1 2 2 2 0 0 0 0 0 2 0 0 0 2 2]
(32, 128, 128, 3) [2 2 0 2 1 2 0 2 0 2 2 2 0 1 0 2 0 0 2 0 1 2 0 0 0 2 0 0 2 0 2 2]


## Performance optimization
- AUTOTUNE is TensorFlow’s way of automatically optimizing CPU/GPU pipeline performance so don’t need to manually tune threads or prefetch sizes
- Decides how many CPU threads to use
- Controls parallel execution in .map()
- Decides how many batches to load ahead of time
- Keep GPU/CPU always busy by preparing next batch early
- Ensures GPU is not waiting for data loading
- Remove manual tuning effort no need to set:
  - number of threads
  - prefetch buffer size
  - parallel workers

In [27]:
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

### since we added scaling, augmentations in image preprocessing dont add scaling layer or augmentation layer in CNN architecture